# DSC 540 Data Preparation
## Final Project - API
### Mile Stone IV
#### David Wederstrandt

In [1]:
# Load packages
import pandas as pd
import requests

from rapidfuzz import process, fuzz

In [2]:
# Correct base URL for the new API
base_url = "https://clinicaltrials.gov/api/v2/studies"
    
# Parameters for searching PTSD interventions
params = {
    'query.cond': 'Posttraumatic Stress Disorder',
    # 'query.cond': 'PTSD',  # Condition search
    'query.intr': 'PTSD',  # Intervention search
    'format': 'json',
    'pageSize': 20,  # Number of results per page
    'countTotal': 'true'
}
    
headers = {
    'User-Agent': 'Python Clinical Trials Searcher'
}

In [3]:
def search_ptsd_interventions(base_url, params, headers):
    """Search for PTSD-related clinical trials using the current ClinicalTrials.gov API"""
    
    try:
        response = requests.get(base_url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        # Handle the new data structure
        studies = data.get('studies', [])
        total_count = data.get('totalCount', 0)
        
        print(f"Found {total_count} studies related to PTSD")
        print("=" * 80)
        
        for study in studies:
            protocol_section = study.get('protocolSection', {})
            identification_module = protocol_section.get('identificationModule', {})
            conditions_module = protocol_section.get('conditionsModule', {})
            interventions_module = protocol_section.get('interventionsModule', {})
            
            print(f"NCT ID: {identification_module.get('nctId', 'N/A')}")
            print(f"Title: {identification_module.get('briefTitle', 'N/A')}")
            print(f"Conditions: {', '.join(conditions_module.get('conditions', []))}")
            
            # Extract interventions
            interventions = interventions_module.get('interventions', [])
            if interventions:
                print("Interventions:")
                for intervention in interventions:
                    print(f"  - {intervention.get('name', 'N/A')} ({intervention.get('type', 'N/A')})")
            
            print("-" * 80)
            
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
    except ValueError as e:
        print(f"JSON parsing error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

    return response

In [4]:
response = search_ptsd_interventions(base_url, params, headers)

Found 1571 studies related to PTSD
NCT ID: NCT05240924
Title: ERP to Improve Functioning in Veterans With OCD
Conditions: Obsessive Compulsive Disorder (OCD), Comorbid Post-Traumatic Stress Disorder and OCD
--------------------------------------------------------------------------------
NCT ID: NCT02720016
Title: Cognitive-Behavioral Conjoint Therapy (CBCT) Project
Conditions: Post Traumatic Stress Disorder
--------------------------------------------------------------------------------
NCT ID: NCT06761716
Title: Comparison of Ericksonian Hypnotherapy and CBT in PTSD: A Clinical Trial
Conditions: PTSD
--------------------------------------------------------------------------------
NCT ID: NCT04775524
Title: AWARENESS Trial (AWARENESS) of Storytelling Through Music in Healthcare Workers
Conditions: Stress, Post Traumatic Stress Disorder, Work Related Stress
--------------------------------------------------------------------------------
NCT ID: NCT06467916
Title: The Development of PATH

## Step 1: Replace Headers
#### Purpose: Transform the raw API data to a clean, structured DataFrame with standardized column names

In [5]:
# Extract data from API response and create DataFrame with clean headers

def extract_studies_to_dataframe(response):
    try:
        data = response.json()

        studies = data.get('studies', [])

        # Extract the relevant data for each study
        studies_data = []
        for study in studies:
            protocol_section = study.get('protocolSection', {})
            identification_module = protocol_section.get('identificationModule', {})
            conditions_module = protocol_section.get('conditionsModule', {})
            interventions_module = protocol_section.get('interventionsModule', {})
            arms_module = protocol_section.get('armsInterventionsModule', {})
            status_module = protocol_section.get('statusModule', {})
            design_module = protocol_section.get('designModule', {})
            
            # Extract interventions
            interventions = interventions_module.get('interventions', [])
            # If no interventions found, try armsInterventionsModule
            if not interventions and arms_module:
                interventions = arms_module.get('interventions', [])
            
            # Check if interventions exist and extract data
            if interventions:
                intervention_types = []
                intervention_names = []
                
                for interv in interventions:
                    # Get intervention type
                    interv_type = interv.get('type', '')
                    if interv_type:
                        intervention_types.append(interv_type)
                    
                    # Get intervention name
                    interv_name = interv.get('name', '')
                    if interv_name:
                        intervention_names.append(interv_name)
                
                # Join the lists
                intervention_types_str = ', '.join(intervention_types) if intervention_types else ''
                intervention_names_str = ', '.join(intervention_names) if intervention_names else ''
            else:
                # No interventions found in either location
                intervention_types_str = ''
                intervention_names_str = ''
            
            study_data = {
                'NCT ID': identification_module.get('nctId', ''),
                'Brief Title': identification_module.get('briefTitle', ''),
                'Official Title': identification_module.get('officialTitle', ''),
                'Conditions': ', '.join(conditions_module.get('conditions', [])),
                'Intervention Types': intervention_types_str,
                'Intervention Names': intervention_names_str,
                'Study Status': status_module.get('overallStatus', ''),
                'Study Phase': ', '.join(design_module.get('phases', [])),
                'Start Date': status_module.get('startDateStruct', {}).get('date', ''),
                'Completion Date': status_module.get('completionDateStruct', {}).get('date', '')
            }           
            studies_data.append(study_data)

        # Create the DataFrame
        df = pd.DataFrame(studies_data)

        # Replace the headers with underscores, convert to lower case, and remove special characters
        print("Original columns: ", df.columns.tolist())
        df.columns = [col.strip().replace(' ', '_').lower() for col in df.columns]
        print("Cleaned column names", df.columns.tolist())
        print(f"The DataFrame's Shape: {df.shape}")

        return df

    except Exception as e:
        print(f"Error extracting data: {e}")
        return pd.DataFrame()

In [6]:
studies_df = extract_studies_to_dataframe(response)
studies_df.head(3)

Original columns:  ['NCT ID', 'Brief Title', 'Official Title', 'Conditions', 'Intervention Types', 'Intervention Names', 'Study Status', 'Study Phase', 'Start Date', 'Completion Date']
Cleaned column names ['nct_id', 'brief_title', 'official_title', 'conditions', 'intervention_types', 'intervention_names', 'study_status', 'study_phase', 'start_date', 'completion_date']
The DataFrame's Shape: (20, 10)


,nct_id,brief_title,official_title,conditions,intervention_types,intervention_names,study_status,study_phase,start_date,completion_date
0,NCT05240924,ERP to Improve Functioning in Veterans With OCD,Exposure and Response Prevention to Improve Fu...,"Obsessive Compulsive Disorder (OCD), Comorbid ...","BEHAVIORAL, OTHER","Exposure and Response Prevention, Stress Manag...",RECRUITING,NA,2022-10-03,2026-05-31
1,NCT02720016,Cognitive-Behavioral Conjoint Therapy (CBCT) P...,An Integrative Technology Approach to Home-bas...,Post Traumatic Stress Disorder,"BEHAVIORAL, BEHAVIORAL, BEHAVIORAL","CBCT-Home Based (CBCT-HB), CBCT-Office Based (...",COMPLETED,NA,2016-10-03,2021-03-31
2,NCT06761716,Comparison of Ericksonian Hypnotherapy and CBT...,Neuropsychophysiological Comparison of Erickso...,PTSD,"BEHAVIORAL, BEHAVIORAL","Ericksonian Hypnotherapy, Cognitive Behavioral...",COMPLETED,NA,2024-11-15,2025-04-15


## Step 2: Format Data into a More Readable Format
##### Purpose: Convert raw data types into appropriate formats and standardize text values to improve data quality, consistency, and readability.

In [7]:
print(studies_df.dtypes)

nct_id                object
brief_title           object
official_title        object
conditions            object
intervention_types    object
intervention_names    object
study_status          object
study_phase           object
start_date            object
completion_date       object
dtype: object


In [8]:
# Convert data types, clean dates, and standardize text values

def format_dataframe(df):
    """
    Format DataFrame columns to appropriate data types and readable formats
    """
    # Create a copy to avoid modifying the original
    formatted_df = df.copy()
    
    # Convert date columns to datetime format
    date_columns = ['start_date', 'completion_date']
    for col in date_columns:
        if col in formatted_df.columns:
            print(f"Converting {col} to datetime format...")
            # Convert to datetime, handling various date formats
            formatted_df[col] = pd.to_datetime(formatted_df[col], errors='coerce')
    
    # Clean and standardize text columns
    text_columns = ['brief_title', 'official_title', 'conditions', 'intervention_names', 'study_status']
    for col in text_columns:
        if col in formatted_df.columns:
            print(f"Cleaning and standardizing {col}...")
            # Convert to string, strip whitespace, handle empty values
            formatted_df[col] = formatted_df[col].astype(str).str.strip().str.title()
            formatted_df[col] = formatted_df[col].replace(['', 'Nan', 'None', 'Na'], 'Not Specified')
    
    # Convert study_phase to categorical for better analysis
    if 'study_phase' in formatted_df.columns:
        print("Converting study_phase to categorical...")
        formatted_df['study_phase'] = formatted_df['study_phase'].replace(['', 'nan', 'na'], 'Not Available')
        formatted_df['study_phase'] = formatted_df['study_phase'].astype('category')
    
    # Standardize intervention types - convert to title case for consistency
    if 'intervention_types' in formatted_df.columns:
        print("Standardizing intervention_types...")
        formatted_df['intervention_types'] = formatted_df['intervention_types'].str.title()
        formatted_df['intervention_types'] = formatted_df['intervention_types'].replace(['', 'Nan', 'Na'], 'Not Specified')
    
    # Clean NCT ID format
    if 'nct_id' in formatted_df.columns:
        print("Formatting NCT IDs...")
        formatted_df['nct_id'] = formatted_df['nct_id'].str.upper().str.strip()
    
    return formatted_df

In [9]:
fotmatted_studies_df = format_dataframe(studies_df)
fotmatted_studies_df.head()

Converting start_date to datetime format...
Converting completion_date to datetime format...
Cleaning and standardizing brief_title...
Cleaning and standardizing official_title...
Cleaning and standardizing conditions...
Cleaning and standardizing intervention_names...
Cleaning and standardizing study_status...
Converting study_phase to categorical...
Standardizing intervention_types...
Formatting NCT IDs...


,nct_id,brief_title,official_title,conditions,intervention_types,intervention_names,study_status,study_phase,start_date,completion_date
0,NCT05240924,Erp To Improve Functioning In Veterans With Ocd,Exposure And Response Prevention To Improve Fu...,"Obsessive Compulsive Disorder (Ocd), Comorbid ...","Behavioral, Other","Exposure And Response Prevention, Stress Manag...",Recruiting,NA,2022-10-03,2026-05-31
1,NCT02720016,Cognitive-Behavioral Conjoint Therapy (Cbct) P...,An Integrative Technology Approach To Home-Bas...,Post Traumatic Stress Disorder,"Behavioral, Behavioral, Behavioral","Cbct-Home Based (Cbct-Hb), Cbct-Office Based (...",Completed,NA,2016-10-03,2021-03-31
2,NCT06761716,Comparison Of Ericksonian Hypnotherapy And Cbt...,Neuropsychophysiological Comparison Of Erickso...,Ptsd,"Behavioral, Behavioral","Ericksonian Hypnotherapy, Cognitive Behavioral...",Completed,NA,2024-11-15,2025-04-15
3,NCT04775524,Awareness Trial (Awareness) Of Storytelling Th...,An Arts-Based Intervention With Healthcare Wor...,"Stress, Post Traumatic Stress Disorder, Work R...","Behavioral, Behavioral","Storytelling Through Music (Stm), Wait List / ...",Completed,NA,2021-03-01,2024-08-31
4,NCT06467916,"The Development Of Path, A Program To Support ...","The Development Of Path, A Program To Support ...",Perinatal Mental Health,Behavioral,Behavioral Treatment,Recruiting,NA,2025-04-22,2026-04-30


## Step 3 & 4: Identify outliers, bad data, and duplicates
##### Purpose: Detect and address data quality issues, outliers, missing values, and inconsistencies that could compromise analysis accuracy and reliability.

In [10]:
# Check for missing data
def check_for_missing_data(df):
    """
    Check for missing data in a dataset
    """
    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100
    missing_df = pd.DataFrame({
        "missing_count": missing_data,
        "missing_percentage": missing_percent
    })
    print(missing_df[missing_df['missing_count'] > 0])
    

In [11]:
def handle_missing_dates(df):
    """
    Handle missing start and completion date values
    """
    date_cleaned_df = df.copy()

    # Handle missing start_date values
    if 'start_date' in date_cleaned_df.columns:
        # keep all NaT or flag them for future investigation
        missing_start_masked = date_cleaned_df['start_date'].isna()

        if missing_start_masked.any():
            date_cleaned_df['start_date_missing'] = missing_start_masked

    # Handle missing completion_dates
    if 'completion_date' in date_cleaned_df.columns:
        missing_comp_masked = date_cleaned_df['completion_date'].isna()

        if missing_comp_masked.any():
            date_cleaned_df['completion_date_missing'] = missing_comp_masked

        # completion date will be missing for ongoing studies
        if 'study_status' in date_cleaned_df.columns:
                ongoing_statuses = ['Recruiting', 'Active, not recruiting', 'Not yet recruiting', 'Enrolling by invitation']
                ongoing_studies = date_cleaned_df['study_status'].isin(ongoing_statuses)
                expected_missing = ongoing_studies & missing_comp_masked  

    # Handle dae logic errors (if completion date comes before the start date
    if 'start_date' in date_cleaned_df.columns and 'completion_date' in date_cleaned_df.columns:
        both_dates_mask = date_cleaned_df['start_date'].notna() & date_cleaned_df['completion_date'].notna()
        logic_error_mask = (date_cleaned_df['completion_date'] < date_cleaned_df['start_date']) & both_dates_mask
        
        if logic_error_mask.any():
            date_cleaned_df['date_logic_error'] = logic_error_mask

            problem_studies = date_cleaned_df[logic_error_mask][['nct_id', 'brief_title', 'start_date', 'completion_date']]
            print(problem_studies)

    flag_columns = [col for col in date_cleaned_df.columns if col.endswith('_missing') or col.endswith('_error')]
    if flag_columns:
        print(f"Flag columns created: {flag_columns}")
        for col in flag_columns:
            print(f" {col}: {date_cleaned_df[col].sum()} records flagged")

    return date_cleaned_df   
        

In [12]:
def fill_missing_start_dates_with_median(df):
    """Fill missing start_date values with the median start_date"""
    
    # Create a copy to avoid modifying the original
    filled_df = df.copy()
    
    # Check if we have the required columns
    if 'start_date' not in filled_df.columns:
        return filled_df
    
    # Check for both missing values and NaT values
    missing_or_nat_mask = filled_df['start_date'].isna() | filled_df['start_date'].isnull()
    
    if 'start_date_missing' not in filled_df.columns:
        filled_df['start_date_missing'] = missing_or_nat_mask
    
    # Count missing values before filling (including NaT)
    missing_count_before = missing_or_nat_mask.sum()
    
    if missing_count_before == 0:
        return filled_df
    
    # Calculate median start date from non-missing, non-NaT values
    valid_start_dates = filled_df[~missing_or_nat_mask]['start_date']
    
    if len(valid_start_dates) == 0:
        return filled_df
    
    median_start_date = valid_start_dates.median()
    
    # Fill missing start dates (including NaT) with median
    mask_to_fill = missing_or_nat_mask
    filled_df.loc[mask_to_fill, 'start_date'] = median_start_date
    
    # Update the missing flag (they're no longer missing)
    filled_df.loc[mask_to_fill, 'start_date_missing'] = False
    
    # Add a flag to track which values were filled with median
    filled_df['start_date_filled_with_median'] = mask_to_fill
    
    return filled_df


In [13]:
def check_date_outliers(df):
    """
    Check for date-related outliers and inconsistencies
    """
    
    # Check start_date column
    if 'start_date' in df.columns:
        # Check for missing start dates
        missing_start = df['start_date'].isna().sum()
        print(f"Missing start dates: {missing_start} ({missing_start/len(df)*100:.1f}%)")
        
        # Check for future dates or very old dates
        current_year = pd.Timestamp.now().year
        if df['start_date'].notna().any():
            start_years = df['start_date'].dt.year
            future_dates = start_years > current_year
            very_old_dates = start_years < 1990
            
            print(f"Studies with future start dates: {future_dates.sum()}")
            print(f"Studies with start dates before 1990: {very_old_dates.sum()}")
            
            if future_dates.any():
                print("Future start dates found:")
                print(df[future_dates][['nct_id', 'brief_title', 'start_date']])
    
    # Check completion_date column
    if 'completion_date' in df.columns:
        # Check for missing completion dates
        missing_completion = df['completion_date'].isna().sum()
        print(f"Missing completion dates: {missing_completion} ({missing_completion/len(df)*100:.1f}%)")
        
        # Check for completion dates before start dates
        if 'start_date' in df.columns:
            date_logic_errors = (df['completion_date'] < df['start_date']).sum()
            print(f"Studies with completion date before start date: {date_logic_errors}")
            
            if date_logic_errors > 0:
                print("Date logic errors found:")
                error_mask = df['completion_date'] < df['start_date']
                print(df[error_mask][['nct_id', 'brief_title', 'start_date', 'completion_date']])


In [14]:
def check_data_qual(df):
    """
    Check for suspicious data patterns
    """
    if 'brief_title' in df.columns:
        short_titles = df['brief_title'].str.len() < 10
        print(f'Studies with very short title (<10 chars): {short_titles.sum()}')

In [15]:
def check_for_dups(df):
    """
    This function will check for duplicate NCT IDs, as they should be unique
    """
    if 'nct_id' in df.columns:
        dup_nct = df['nct_id'].duplicated()
        dup_count = dup_nct.sum()
        if dup_count >= 1:
            print(f"Duplicate NCT IDs: {dup_nct}")
        else:
            print("No duplicate NCT IDs found.")

In [16]:
def check_intervention_quality(df):
    """
    Check intervention data quality
    """
    if 'intervention_types' in df.columns:
        no_intervention = (df['intervention_types'] == 'Not Specified') | (df['intervention_types'] == '')
        print(f"Studies with no intervention type specified: {no_intervention.sum()}")
    
    if 'intervention_names' in df.columns:
        no_intervention_name = (df['intervention_names'] == 'Not Specified') | (df['intervention_names'] == '')
        print(f"Studies with no intervention name specified: {no_intervention_name.sum()}")

In [17]:
# This function will call all of the 
def identify_outliers_and_bad_data(df):
    """
    Identify outliers, missing data, and potential data quality issues
    """
    
    print("Step 3: Identifying outliers and bad data")
    print("="*60)

    # check for missing data and handle
    check_for_missing_data(df)
    df = handle_missing_dates(df)
    print("\n" + "-"*60)

    # Fill in missing start dates
    df = fill_missing_start_dates_with_median(df)
    print("\n" + "-"*60)

    
    check_date_outliers(df)
    print("\n" + "-"*60)
    
    check_data_qual(df)
    print("\n" + "-"*60)
    
    check_for_dups(df)
    print("\n" + "-"*60)
    
    check_intervention_quality(df)
    
    print("\n" + "="*60)
    print("Data quality summary complete")
    
    return df


I am missing over 40% of start and completion dates, this is a lot so I will fixed them instead of dropping

In [18]:
step3_df = identify_outliers_and_bad_data(fotmatted_studies_df)
step3_df.head()

Step 3: Identifying outliers and bad data
                 missing_count  missing_percentage
start_date                   6                30.0
completion_date              8                40.0
Flag columns created: ['start_date_missing', 'completion_date_missing']
 start_date_missing: 6 records flagged
 completion_date_missing: 8 records flagged

------------------------------------------------------------

------------------------------------------------------------
Missing start dates: 0 (0.0%)
Studies with future start dates: 0
Studies with start dates before 1990: 0
Missing completion dates: 8 (40.0%)
Studies with completion date before start date: 0

------------------------------------------------------------
Studies with very short title (<10 chars): 0

------------------------------------------------------------
No duplicate NCT IDs found.

------------------------------------------------------------
Studies with no intervention type specified: 0
Studies with no intervention 

,nct_id,brief_title,official_title,conditions,intervention_types,intervention_names,study_status,study_phase,start_date,completion_date,start_date_missing,completion_date_missing,start_date_filled_with_median
0,NCT05240924,Erp To Improve Functioning In Veterans With Ocd,Exposure And Response Prevention To Improve Fu...,"Obsessive Compulsive Disorder (Ocd), Comorbid ...","Behavioral, Other","Exposure And Response Prevention, Stress Manag...",Recruiting,NA,2022-10-03,2026-05-31,False,False,False
1,NCT02720016,Cognitive-Behavioral Conjoint Therapy (Cbct) P...,An Integrative Technology Approach To Home-Bas...,Post Traumatic Stress Disorder,"Behavioral, Behavioral, Behavioral","Cbct-Home Based (Cbct-Hb), Cbct-Office Based (...",Completed,NA,2016-10-03,2021-03-31,False,False,False
2,NCT06761716,Comparison Of Ericksonian Hypnotherapy And Cbt...,Neuropsychophysiological Comparison Of Erickso...,Ptsd,"Behavioral, Behavioral","Ericksonian Hypnotherapy, Cognitive Behavioral...",Completed,NA,2024-11-15,2025-04-15,False,False,False
3,NCT04775524,Awareness Trial (Awareness) Of Storytelling Th...,An Arts-Based Intervention With Healthcare Wor...,"Stress, Post Traumatic Stress Disorder, Work R...","Behavioral, Behavioral","Storytelling Through Music (Stm), Wait List / ...",Completed,NA,2021-03-01,2024-08-31,False,False,False
4,NCT06467916,"The Development Of Path, A Program To Support ...","The Development Of Path, A Program To Support ...",Perinatal Mental Health,Behavioral,Behavioral Treatment,Recruiting,NA,2025-04-22,2026-04-30,False,False,False


## Step 5: Fix casing or inconsistent values
#### Purpose: Detect and address casting issues and inconsistent values that could compromise analysis accuracy and reliability.

In [19]:
def standardize_study_status(df):
    """
    Standardize study status values
    """
    status_mapping = {
        'recruiting': 'Recruiting',
        'RECRUITING': 'Recruiting',
        'active, not recruiting': 'Active, not recruiting',
        'ACTIVE, NOT RECRUITING': 'Active, not recruiting',
        'completed': 'Completed',
        'COMPLETED': 'Completed',
        'terminated': 'Terminated',
        'TERMINATED': 'Terminated',
        'withdrawn': 'Withdrawn',
        'WITHDRAWN': 'Withdrawn',
        'suspended': 'Suspended',
        'SUSPENDED': 'Suspended',
        'not yet recruiting': 'Not yet recruiting',
        'NOT YET RECRUITING': 'Not yet recruiting',
        'enrolling by invitation': 'Enrolling by invitation',
        'ENROLLING BY INVITATION': 'Enrolling by invitation'
    }
    
    if 'study_status' in df.columns:
        # Apply mapping and handle case variations
        df['study_status'] = df['study_status'].str.strip()
        df['study_status'] = df['study_status'].replace(status_mapping)
    
    return df

In [20]:
mapping_df = standardize_study_status(step3_df)

In [21]:
def standardize_intervention_types(df):
    """
    Standardize intervention type values
    """
    type_mapping = {
        'drug': 'Drug',
        'DRUG': 'Drug',
        'behavioral': 'Behavioral',
        'BEHAVIORAL': 'Behavioral',
        'procedure': 'Procedure',
        'PROCEDURE': 'Procedure',
        'device': 'Device',
        'DEVICE': 'Device',
        'other': 'Other',
        'OTHER': 'Other',
        'biological': 'Biological',
        'BIOLOGICAL': 'Biological',
        'dietary supplement': 'Dietary Supplement',
        'DIETARY SUPPLEMENT': 'Dietary Supplement',
        'radiation': 'Radiation',
        'RADIATION': 'Radiation'
    }
    
    if 'intervention_types' in df.columns:
        # Handle multiple intervention types separated by commas
        def clean_intervention_types(value):
            if pd.isna(value) or value == 'Not Specified':
                return value
            
            # Split by comma, clean each type, then rejoin
            types = [t.strip() for t in str(value).split(',')]
            cleaned_types = []
            
            for t in types:
                if t.lower() in type_mapping:
                    cleaned_types.append(type_mapping[t.lower()])
                else:
                    # Apply title case for unknown types
                    cleaned_types.append(t.title())
            
            return ', '.join(cleaned_types)
        
        df['intervention_types'] = df['intervention_types'].apply(clean_intervention_types)
    
    return df


In [22]:
mapping_df = standardize_intervention_types(mapping_df)

In [23]:
def standardize_text_fields(df):
    """
    Standardize general text fields for consistent casing
    """
    text_fields = ['brief_title', 'official_title', 'conditions', 'intervention_names']
    
    for field in text_fields:
        if field in df.columns:
            # Fix common casing issues while preserving acronyms
            def fix_casing(text):
                if pd.isna(text) or text == 'Not Specified':
                    return text
                
                text = str(text).strip()
                
                # Common acronyms that should stay uppercase
                acronyms = ['PTSD', 'CBT', 'EMDR', 'TBI', 'FDA', 'NIH', 'VA', 'DOD', 'HIV', 'AIDS', 'COVID', 'MRI', 'fMRI', 'EEG', 'DSM', 'ICD']
                
                # Split into words and process
                words = text.split()
                processed_words = []
                
                for word in words:
                    # Remove punctuation for checking but keep it for output
                    clean_word = ''.join(c for c in word if c.isalnum())
                    
                    if clean_word.upper() in acronyms:
                        # Replace the alphabetic part with uppercase while keeping punctuation
                        new_word = ''
                        for c in word:
                            if c.isalpha():
                                new_word += c.upper()
                            else:
                                new_word += c
                        processed_words.append(new_word)
                    elif len(clean_word) > 1:
                        # Apply title case for regular words
                        processed_words.append(word.capitalize())
                    else:
                        # Keep single characters as-is
                        processed_words.append(word)
                
                return ' '.join(processed_words)
            
            df[field] = df[field].apply(fix_casing)
    
    return df


In [24]:
mapping_df = standardize_text_fields(mapping_df)

## Step 6: Conduct Fuzzy Matching

In [25]:
def fuzzy_match_interventions(df, threshold=80):
    """
    Perform fuzzy matching on intervention names to identify and consolidate similar entries
    """
    
    # Create a copy to avoid modifying the original
    fuzzy_df = df.copy()
    
    if 'intervention_names' not in fuzzy_df.columns:
        print("No intervention_names column found for fuzzy matching")
        return fuzzy_df
    
    # Get unique intervention names (excluding empty/not specified)
    unique_interventions = fuzzy_df['intervention_names'].dropna()
    unique_interventions = unique_interventions[unique_interventions != 'Not Specified']
    unique_interventions = unique_interventions[unique_interventions != '']
    unique_interventions = unique_interventions.unique()
    
    print(f"Found {len(unique_interventions)} unique intervention names for fuzzy matching")
    
    # Create mapping dictionary for similar interventions
    intervention_mapping = {}
    processed = set()
    
    for intervention in unique_interventions:
        if intervention in processed:
            continue
            
        # Find similar interventions
        matches = process.extract(intervention, unique_interventions, scorer=fuzz.ratio, limit=None)
        similar_interventions = [match[0] for match in matches if match[1] >= threshold and match[0] not in processed]
        
        if len(similar_interventions) > 1:
            # Use the most common or first intervention as the canonical form
            canonical = similar_interventions[0]
            print(f"Grouping similar interventions under '{canonical}':")
            
            for similar in similar_interventions:
                intervention_mapping[similar] = canonical
                processed.add(similar)
                if similar != canonical:
                    print(f"  - '{similar}' -> '{canonical}'")
    
    # Apply the mapping
    if intervention_mapping:
        fuzzy_df['intervention_names_original'] = fuzzy_df['intervention_names'].copy()
        fuzzy_df['intervention_names'] = fuzzy_df['intervention_names'].replace(intervention_mapping)
        print(f"\nApplied {len(intervention_mapping)} intervention mappings")
    else:
        print("No similar intervention names found for consolidation")
    
    return fuzzy_df


In [26]:
fuzzy_df = fuzzy_match_interventions(mapping_df)

Found 20 unique intervention names for fuzzy matching
No similar intervention names found for consolidation


In [27]:
def fuzzy_match_conditions(df, threshold=85):
    """
    Perform fuzzy matching on condition names to identify and consolidate similar entries
    """
    
    # Create a copy to avoid modifying the original
    df = df.copy()
    
    if 'conditions' not in df.columns:
        print("No conditions column found for fuzzy matching")
        return df
    
    # Get unique conditions (excluding empty/not specified)
    unique_conditions = df['conditions'].dropna()
    unique_conditions = unique_conditions[unique_conditions != 'Not Specified']
    unique_conditions = unique_conditions[unique_conditions != '']
    unique_conditions = unique_conditions.unique()
    
    print(f"Found {len(unique_conditions)} unique condition names for fuzzy matching")
    
    # Create mapping dictionary for similar conditions
    condition_mapping = {}
    processed = set()
    
    for condition in unique_conditions:
        if condition in processed:
            continue
            
        # Find similar conditions
        matches = process.extract(condition, unique_conditions, scorer=fuzz.ratio, limit=None)
        similar_conditions = [match[0] for match in matches if match[1] >= threshold and match[0] not in processed]
        
        if len(similar_conditions) > 1:
            # Use the most common or first condition as the canonical form
            canonical = similar_conditions[0]
            print(f"Grouping similar conditions under '{canonical}':")
            
            for similar in similar_conditions:
                condition_mapping[similar] = canonical
                processed.add(similar)
                if similar != canonical:
                    print(f"  - '{similar}' -> '{canonical}'")
    
    # Apply the mapping
    if condition_mapping:
        df['conditions_original'] = df['conditions'].copy()
        df['conditions'] = df['conditions'].replace(condition_mapping)
        print(f"\nApplied {len(condition_mapping)} condition mappings")
    else:
        print("No similar condition names found for consolidation")
    
    return df


In [28]:
final_df = fuzzy_match_conditions(fuzzy_df)
final_df.head()

Found 13 unique condition names for fuzzy matching
Grouping similar conditions under 'Post Traumatic Stress Disorder':
  - 'Posttraumatic Stress Disorder' -> 'Post Traumatic Stress Disorder'
  - 'Post-traumatic Stress Disorder' -> 'Post Traumatic Stress Disorder'

Applied 3 condition mappings


,nct_id,brief_title,official_title,conditions,intervention_types,intervention_names,study_status,study_phase,start_date,completion_date,start_date_missing,completion_date_missing,start_date_filled_with_median,conditions_original
0,NCT05240924,Erp To Improve Functioning In Veterans With Ocd,Exposure And Response Prevention To Improve Fu...,"Obsessive Compulsive Disorder (ocd), Comorbid ...","Behavioral, Other","Exposure And Response Prevention, Stress Manag...",Recruiting,NA,2022-10-03,2026-05-31,False,False,False,"Obsessive Compulsive Disorder (ocd), Comorbid ..."
1,NCT02720016,Cognitive-behavioral Conjoint Therapy (cbct) P...,An Integrative Technology Approach To Home-bas...,Post Traumatic Stress Disorder,"Behavioral, Behavioral, Behavioral","Cbct-home Based (cbct-hb), Cbct-office Based (...",Completed,NA,2016-10-03,2021-03-31,False,False,False,Post Traumatic Stress Disorder
2,NCT06761716,Comparison Of Ericksonian Hypnotherapy And CBT...,Neuropsychophysiological Comparison Of Erickso...,PTSD,"Behavioral, Behavioral","Ericksonian Hypnotherapy, Cognitive Behavioral...",Completed,NA,2024-11-15,2025-04-15,False,False,False,PTSD
3,NCT04775524,Awareness Trial (awareness) Of Storytelling Th...,An Arts-based Intervention With Healthcare Wor...,"Stress, Post Traumatic Stress Disorder, Work R...","Behavioral, Behavioral","Storytelling Through Music (stm), Wait List / ...",Completed,NA,2021-03-01,2024-08-31,False,False,False,"Stress, Post Traumatic Stress Disorder, Work R..."
4,NCT06467916,"The Development Of Path, A Program To Support ...","The Development Of Path, A Program To Support ...",Perinatal Mental Health,Behavioral,Behavioral Treatment,Recruiting,NA,2025-04-22,2026-04-30,False,False,False,Perinatal Mental Health


In [29]:
# Show examples of fuzzy matching results.
print("Fuzzy Matching Results Summary")
print("="*50)

# Show intervention name changes if any
if 'intervention_names_original' in final_df.columns:
    changed_interventions = final_df[final_df['intervention_names'] != final_df['intervention_names_original']]
    
    if len(changed_interventions) > 0:
        print(f"\nIntervention Names - Examples of Changes:")
        print("-"*40)
        for idx, row in changed_interventions.head(5).iterrows():
            print(f"Original: {row['intervention_names_original']}")
            print(f"Matched:  {row['intervention_names']}")
            print()
    else:
        print("No intervention name changes made through fuzzy matching")

# Show condition changes if any
if 'conditions_original' in final_df.columns:
    changed_conditions = final_df[final_df['conditions'] != final_df['conditions_original']]
    
    if len(changed_conditions) > 0:
        print(f"\nCondition Names - Examples of Changes:")
        print("-"*40)
        for idx, row in changed_conditions.head(5).iterrows():
            print(f"Original: {row['conditions_original']}")
            print(f"Matched:  {row['conditions']}")
            print()
    else:
        print("No condition name changes made through fuzzy matching")

# Show unique counts before and after
print(f"\nData Consolidation Summary:")
print("-"*30)

if 'intervention_names_original' in final_df.columns:
    original_unique = final_df['intervention_names_original'].nunique()
    final_unique = final_df['intervention_names'].nunique()
    print(f"Intervention names - Before: {original_unique}, After: {final_unique}")

if 'conditions_original' in final_df.columns:
    original_unique_cond = final_df['conditions_original'].nunique()
    final_unique_cond = final_df['conditions'].nunique()
    print(f"Conditions - Before: {original_unique_cond}, After: {final_unique_cond}")

print(f"\nFinal cleaned dataset ready for analysis!")
print(f"Shape: {final_df.shape}")
print(f"All data preparation steps completed successfully.")

Fuzzy Matching Results Summary

Condition Names - Examples of Changes:
----------------------------------------
Original: Posttraumatic Stress Disorder
Matched:  Post Traumatic Stress Disorder

Original: Posttraumatic Stress Disorder
Matched:  Post Traumatic Stress Disorder

Original: Posttraumatic Stress Disorder
Matched:  Post Traumatic Stress Disorder

Original: Posttraumatic Stress Disorder
Matched:  Post Traumatic Stress Disorder

Original: Post-traumatic Stress Disorder
Matched:  Post Traumatic Stress Disorder


Data Consolidation Summary:
------------------------------
Conditions - Before: 13, After: 11

Final cleaned dataset ready for analysis!
Shape: (20, 14)
All data preparation steps completed successfully.


## Ethical Implications of Data Wrangling

### Clinical Trials Data Ethics and Regulatory Considerations

The data wrangling process conducted on PTSD clinical trial data from ClinicalTrials.gov raises important ethical considerations requiring careful attention. **Data transformations performed** included standardizing headers, converting data types, filling missing start dates with median values, standardizing text casing, and conducting fuzzy matching to consolidate similar intervention and condition names. **Regulatory compliance** is paramount as this data falls under FDA regulations (21 CFR Part 11) and HIPAA guidelines, requiring strict data integrity principles and audit trails. **Transformation risks** include temporal bias from median date imputation that could misrepresent study timelines, and fuzzy matching, potentially grouping distinct treatments with different efficacy profiles. **Critical assumptions** included treating missing completion dates as ongoing studies, assuming similar-sounding interventions are equivalent approaches, and standardizing terminology without verification. **Data credibility** is well-established as ClinicalTrials.gov is the official FDA-mandated registry providing publicly accessible, ethically obtained data through proper research protocols and informed consent. **Mitigation strategies** include maintaining comprehensive audit trails by preserving original values, implementing conservative similarity thresholds (80-85%), clearly documenting assumptions, validating groupings through manual review, establishing data governance protocols requiring domain expert review, and ensuring downstream analyses acknowledge limitations and potential biases, particularly when results might influence clinical decision-making or treatment recommendations.

In [30]:
final_df.reset_index()
final_df.to_csv('data/api_final_data.csv')

In [31]:
final_df

,nct_id,brief_title,official_title,conditions,intervention_types,intervention_names,study_status,study_phase,start_date,completion_date,start_date_missing,completion_date_missing,start_date_filled_with_median,conditions_original
0,NCT05240924,Erp To Improve Functioning In Veterans With Ocd,Exposure And Response Prevention To Improve Fu...,"Obsessive Compulsive Disorder (ocd), Comorbid ...","Behavioral, Other","Exposure And Response Prevention, Stress Manag...",Recruiting,NA,2022-10-03,2026-05-31,False,False,False,"Obsessive Compulsive Disorder (ocd), Comorbid ..."
1,NCT02720016,Cognitive-behavioral Conjoint Therapy (cbct) P...,An Integrative Technology Approach To Home-bas...,Post Traumatic Stress Disorder,"Behavioral, Behavioral, Behavioral","Cbct-home Based (cbct-hb), Cbct-office Based (...",Completed,NA,2016-10-03,2021-03-31,False,False,False,Post Traumatic Stress Disorder
2,NCT06761716,Comparison Of Ericksonian Hypnotherapy And CBT...,Neuropsychophysiological Comparison Of Erickso...,PTSD,"Behavioral, Behavioral","Ericksonian Hypnotherapy, Cognitive Behavioral...",Completed,NA,2024-11-15,2025-04-15,False,False,False,PTSD
3,NCT04775524,Awareness Trial (awareness) Of Storytelling Th...,An Arts-based Intervention With Healthcare Wor...,"Stress, Post Traumatic Stress Disorder, Work R...","Behavioral, Behavioral","Storytelling Through Music (stm), Wait List / ...",Completed,NA,2021-03-01,2024-08-31,False,False,False,"Stress, Post Traumatic Stress Disorder, Work R..."
4,NCT06467916,"The Development Of Path, A Program To Support ...","The Development Of Path, A Program To Support ...",Perinatal Mental Health,Behavioral,Behavioral Treatment,Recruiting,NA,2025-04-22,2026-04-30,False,False,False,Perinatal Mental Health
5,NCT03238924,Prolonged Exposure And Oxytocin,Augmenting Prolonged Exposure Therapy For PTSD...,PTSD,"Drug, Drug","Oxytocin, Placebos",Completed,PHASE2,2015-01-01,2016-12-31,False,False,False,PTSD
6,NCT06535516,Resilient Roots: Supporting Youth And Families,Transform: Translational Research That Adapts ...,Post Traumatic Stress Disorder,"Behavioral, Behavioral",Trauma-focused Cognitive Behavioral Therapy (t...,Recruiting,NA,2024-07-15,2028-08-01,False,False,False,Posttraumatic Stress Disorder
7,NCT02519296,A Fmri Study Of The Treatment Of Danish Vetera...,A Fmri Study Of The Treatment Of Danish Vetera...,"Stress Disorders, Post-traumatic","Behavioral, Behavioral, Device","Prolonged Exposure Therapy, Observation, Fmri",Unknown,NA,2022-11-17,NaT,False,True,True,"Stress Disorders, Post-traumatic"
8,NCT00855816,Post Traumatic Stress Disorder (PTSD) Hyperaro...,PTSD Hyperarousal Symptoms Treated With Physio...,"Stress Disorders, Post-traumatic",Behavioral,Breathing Training,Completed,"PHASE2, PHASE3",2022-11-17,NaT,False,True,True,"Stress Disorders, Post-traumatic"
9,NCT07065396,Virtual Reality And Cranial Nerve Neuromodulat...,Virtual Reality And Cranial Nerve Neuromodulat...,"Chronic Pain, PTSD - Post Traumatic Stress Dis...","Procedure, Procedure, Procedure","Virtual Reality, Cranial Nerve Non-invasive Ne...",Not_Yet_Recruiting,NA,2022-11-17,NaT,False,True,True,"Chronic Pain, PTSD - Post Traumatic Stress Dis..."
